In [24]:
import sys, os
sys.path.append(os.path.abspath('..'))

In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import copy
from sklearn.model_selection import train_test_split

from src.models_numpy.dnn.neuralnet import NeuralNetwork
from src.models_numpy.dnn.layers import DenseLayer, DropoutLayer, BatchNormalizationLayer
from src.models_numpy.dnn.activation import ReLUActivation, SoftmaxActivation
from src.models_numpy.dnn.losses import CategoricalCrossEntropy
from src.models_numpy.dnn.metrics import accuracy
from src.models_numpy.dnn.dataset import Dataset
from src.data_processing import clean_text
from src.vectorizer import create_vectorizer
from src.models_numpy.dnn.optimizer import AdamOptimizer, SGDOptimizer

In [26]:
import nltk
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("punkt")

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

## Preprocessamento
Aplicamos a função `clean_text` com NLTK e dividimos em treino/validação (80/20).

In [27]:
df = pd.read_csv("../data/processed/LabelsParaHiper.csv", sep=",")

df.columns = df.columns.str.strip()
required_cols = {"Text", "Label"}
if not required_cols.issubset(df.columns):
    raise ValueError(f"CSV tem de conter as colunas {required_cols}. Colunas encontradas: {set(df.columns)}")

label_map = {label: i for i, label in enumerate(sorted(df['Label'].unique()))}
df['label_id'] = df['Label'].map(label_map)

df['text_clean'] = df['Text'].apply(clean_text)

X_train, X_val, y_train, y_val = train_test_split(
    df[['Text', 'text_clean']], df['label_id'], test_size=0.2, random_state=42, stratify=df['label_id']
)

num_classes = len(label_map)
y_train_oh = np.eye(num_classes)[y_train.values]
y_val_oh = np.eye(num_classes)[y_val.values]

print(f"Treino: {X_train.shape[0]} amostras")
print(f"Validação: {X_val.shape[0]} amostras")
print(f"Número de classes: {num_classes}")

Treino: 34080 amostras
Validação: 8520 amostras
Número de classes: 5


## Espaço de Parâmetros para Random Search
- Loss functions (CategoricalCrossEntropy, FocalLossMulticlass)
- Activation functions (ReLU, Sigmoid)

In [35]:
param_grid = {
    'loss_function':       [('CategoricalCrossEntropy', CategoricalCrossEntropy)],
    'activation_function': [('ReLU', ReLUActivation)],

    'vectorizer_type':     ["forensic", "stylometric_only", "stylometric"], 
    'char_ngram_range':    [(2, 2), (2, 3)],
    'max_words':           [500, 1000, 2000],

    'hidden_neurons':      [[128, 64], [256, 128], [256, 128, 64]],
    'learning_rate':       [0.001, 0.005],
    'optimizer_type':      ['adam', 'sgd'],
    'epochs':              [30],
    'batch_size':          [32, 64, 128],
    'dropout_rate':        [0.3, 0.4, 0.5],
    'weight_decay':        [0.0000, 0.0001],
    'use_batchnorm':       [True, False],
    'patience':            [20],
}

print(f"Tamanho do grid: {len(param_grid)} parâmetros")
print(f"Combinações totais teóricas: {np.prod([len(v) for v in param_grid.values()])}")

Tamanho do grid: 14 parâmetros
Combinações totais teóricas: 7776


In [36]:
from itertools import product

def train_with_params(params, X_tr, y_train, X_te, y_val, num_classes, vec):
    if params.get('optimizer_type', 'adam') == 'adam':
        opt = AdamOptimizer(learning_rate=params['learning_rate'])
    else:
        opt = SGDOptimizer(learning_rate=params['learning_rate'], momentum=0.9)
        
    model = NeuralNetwork(
        epochs=params['epochs'],
        batch_size=params['batch_size'],
        optimizer=opt,
        loss=params['loss_function'][1],
        metric=accuracy,
        early_stopping=True,
        patience=params.get('patience', 15),
        verbose=False,
    )
    
    input_dim = X_tr.shape[1]
    hidden_layers = params['hidden_neurons']
    
    model.add(DenseLayer(n_units=hidden_layers[0], input_shape=(input_dim,), 
                         l2_reg=params['weight_decay']))
    if params.get('use_batchnorm', False):
        model.add(BatchNormalizationLayer())
    model.add(params['activation_function'][1]())
    if params['dropout_rate'] > 0:
        model.add(DropoutLayer(rate=params['dropout_rate']))
    
    for i, units in enumerate(hidden_layers[1:]):
        model.add(DenseLayer(n_units=units, l2_reg=params['weight_decay']))
        if params.get('use_batchnorm', False):
            model.add(BatchNormalizationLayer())
        model.add(params['activation_function'][1]())
        if params['dropout_rate'] > 0:
            model.add(DropoutLayer(rate=params['dropout_rate'] / (i+2)))
    
    model.add(DenseLayer(n_units=num_classes, l2_reg=params['weight_decay']))
    model.add(SoftmaxActivation())
    
    dataset = Dataset(X_tr, y_train)
    val_dataset = Dataset(X_te, y_val)
    
    start_time = time.time()
    history = model.fit(dataset, val_dataset=val_dataset)
    training_time = time.time() - start_time
    
    if hasattr(model, 'best_epoch') and model.best_epoch is not None:
        best_epoch_idx = model.best_epoch - 1
        val_acc = history['val_acc'][best_epoch_idx] if 'val_acc' in history and best_epoch_idx < len(history['val_acc']) else 0
        val_loss = history['val_loss'][best_epoch_idx]
    elif 'val_loss' in history and len(history['val_loss']) > 0:
        best_epoch = np.argmin(history['val_loss'])
        val_acc = history['val_acc'][best_epoch] if 'val_acc' in history and best_epoch < len(history['val_acc']) else 0
        val_loss = history['val_loss'][best_epoch]
    else:
        val_acc = 0
        val_loss = float('inf')
    
    return {
        'model': model,
        'vectorizer': vec,
        'history': history,
        'val_acc': val_acc,
        'val_loss': val_loss,
        'training_time': training_time,
        'params': params
    }

In [37]:
print("=== A pré-computar vectorizers ===")
vectorizer_cache = {}

combos = list(product(
    param_grid['vectorizer_type'],
    param_grid['char_ngram_range'],
    param_grid['max_words']
))
print(f"Total de combinações: {len(combos)}")  # 3 × 2 × 3 = 18

for vtype, ngram, maxw in combos:
    key = (vtype, ngram, maxw)
    print(f"  A vetorizar: {key}...")
    t0 = time.time()
    vec = create_vectorizer(vtype, max_words=maxw, ngram_range=ngram)
    X_tr_v = vec.fit_transform(list(X_train['text_clean']), list(X_train['Text']))
    X_te_v = vec.transform(list(X_val['text_clean']), list(X_val['Text']))
    vectorizer_cache[key] = (vec, X_tr_v, X_te_v)
    print(f"    -> {X_tr_v.shape[1]} features em {time.time()-t0:.1f}s")

print("\nCache pronta!")

=== A pré-computar vectorizers ===
Total de combinações: 18
  A vetorizar: ('forensic', (2, 2), 500)...


KeyboardInterrupt: 

## Executar Random Search

In [ ]:
from src.hyperopt import build_random_search
import random

N_ITERATIONS = 20
SEED = 42
random.seed(SEED)


results = []
best_val_acc = -1
best_model = None
best_vectorizer = None
best_history = None

print(f"=== A começar Random Search ({N_ITERATIONS} iterações) ===")

search_gen = build_random_search(param_grid, N_ITERATIONS, SEED)

for i in range(N_ITERATIONS):
    sampled_params = next(search_gen)
    
    print(f"\n[{i+1}/{N_ITERATIONS}] Testando: {sampled_params}")
    
    try:
        key = (
            sampled_params['vectorizer_type'],
            sampled_params['char_ngram_range'],
            sampled_params['max_words']
        )
        vec, X_tr_iter, X_te_iter = vectorizer_cache[key]

        result = train_with_params(
            sampled_params,
            X_tr_iter,
            y_train_oh,
            X_te_iter,
            y_val_oh,
            num_classes,
            vec=vec
        )
        
        results.append({
            'iteration': i+1,
            'val_acc': result['val_acc'],
            'val_loss': result['val_loss'],
            'time': result['training_time'],
            **sampled_params
        })
        
        if result['val_acc'] > best_val_acc:
            best_val_acc = result['val_acc']
            best_model = result['model']
            best_vectorizer = result['vectorizer']
            best_history = result['history']
            print(f"  -> Novo melhor: Val Acc: {best_val_acc:.4f}")
        else:
            print(f"  -> Val Acc: {result['val_acc']:.4f}")
    except Exception as e:
        print(f"  -> Erro: {str(e)}")
        results.append({
            'iteration': i+1,
            'error': str(e),
            **sampled_params
        })

=== A começar Random Search (20 iterações) ===

[1/20] Testando: {'loss_function': ('CategoricalCrossEntropy', <class 'src.models_numpy.dnn.losses.CategoricalCrossEntropy'>), 'activation_function': ('ReLU', <class 'src.models_numpy.dnn.activation.ReLUActivation'>), 'vectorizer_type': 'stylometric', 'char_ngram_range': (2, 3), 'max_words': 500, 'hidden_neurons': [128, 64], 'learning_rate': 0.001, 'optimizer_type': 'adam', 'epochs': 30, 'batch_size': 128, 'dropout_rate': 0.4, 'weight_decay': 0.0, 'use_batchnorm': True, 'patience': 20}


KeyboardInterrupt: 

## Análise de Resultados
Vamos analisar os resultados do random search.

In [31]:
results_df = pd.DataFrame(results)
if 'error' in results_df.columns:
    results_df = results_df[results_df['error'].isna()]

if results_df.empty or 'val_acc' not in results_df.columns:
    print("Nenhum resultado válido com 'val_acc'.")
    print("Verifique os erros da célula de treino (ex.: recursos NLTK em falta).")
else:
    print("=== Melhores Resultados ===")
    top_results = results_df.sort_values('val_acc', ascending=False).head(5)
    print(top_results[['iteration', 'val_acc', 'val_loss', 'loss_function', 'activation_function', 
                       'learning_rate', 'max_words', 'hidden_neurons']])

    plt.figure(figsize=(10, 6))
    results_df['val_acc'].hist(bins=20, edgecolor='black', alpha=0.7)
    plt.title('Distribuição de Validation Accuracy')
    plt.xlabel('Accuracy')
    plt.ylabel('Frequência')
    plt.axvline(best_val_acc, color='red', linestyle='--', label=f'Melhor: {best_val_acc:.4f}')
    plt.legend()
    plt.show()

    print("\n=== Correlações com Accuracy ===")
    numeric_cols = results_df.select_dtypes(include=[np.number]).columns
    if 'val_acc' in numeric_cols:
        correlations = results_df[numeric_cols].corr()['val_acc'].sort_values(ascending=False)
        print(correlations.head(10))

Nenhum resultado válido com 'val_acc'.
Verifique os erros da célula de treino (ex.: recursos NLTK em falta).


In [32]:
if best_model:
    print(f"Melhor validation accuracy: {best_val_acc:.4f}")
    print(f"Número total de modelos testados: {len(results_df)}")
    print("\nMelhores hiperparâmetros:")
    best_params = results_df.loc[results_df['val_acc'].idxmax()]
    for key, value in best_params.items():
        if key not in ['iteration', 'val_acc', 'val_loss', 'time', 'error']:
            print(f"  {key}: {value}")

## Validação com Dataset Exemplos

In [33]:
if best_model:
    df_test = pd.read_csv("../data/raw/dataset-exemplos.csv", sep=";")
    
    df_test['label_id'] = df_test['Label'].map(label_map)
    df_test['text_clean'] = df_test['Text'].apply(clean_text)
    
    X_test = best_vectorizer.transform(
        list(df_test['text_clean']),
        list(df_test['Text'])
    )
    class_counts = df['label_id'].value_counts().sort_index().values
    class_weights = len(df) / (num_classes * class_counts)
    class_weights = class_weights / class_weights.sum()
    y_pred_probs = best_model.forward_propagation(X_test, training=False)
    y_pred_weighted = y_pred_probs * class_weights
    y_pred = np.argmax(y_pred_weighted, axis=1)
    y_true = df_test['label_id'].values
    
    print("=== Validação com Dataset Exemplos ===")
    print(f"Número de amostras de teste: {len(df_test)}")
    print(f"Distribuição de classes:\n{df_test['Label'].value_counts()}")
    print()
    
    from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
    
    test_acc = accuracy_score(y_true, y_pred)
    print(f"Accuracy no dataset exemplos: {test_acc:.4f}")
    print(f"Accuracy na validação interna: {best_val_acc:.4f}")
    print(f"Diferença: {test_acc - best_val_acc:.4f}")
    print()
    
    print("=== Relatório de Classificação (Dataset Exemplos) ===")
    print(classification_report(y_true, y_pred, target_names=list(label_map.keys())))
    
    plt.figure(figsize=(10, 8))
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=label_map.keys(), yticklabels=label_map.keys())
    plt.title('Matriz de Confusão - Dataset Exemplos')
    plt.xlabel('Previsto')
    plt.ylabel('Real')
    plt.show()
else:
    print("Nenhum modelo treinado. Execute as células anteriores primeiro.")

#     # Salvar melhor modelo
#     best_model.save("../saved_models/DNN_best_model.npz")
#     print("\nMelhor modelo salvo em '../saved_models/DNN_best_model.npz'")
    
#     # Salvar também o vectorizer
#     import pickle
#     with open('../saved_models/DNN_best_vectorizer.pkl', 'wb') as f:
#         pickle.dump(best_vectorizer, f)
#     print("Vectorizer salvo em '../saved_models/DNN_best_vectorizer.pkl'")
# else:
#     print("Nenhum modelo treinado com sucesso.")

Nenhum modelo treinado. Execute as células anteriores primeiro.
